# [M5] CHIẾN THUẬT THI OLYMPIC AI
## Buổi S02: Feature Engineering Thực Chiến, K-Fold Validation & Ensembling

---

### Metadata Khóa Học
* **Module:** M5 — CHIẾN THUẬT THI OLYMPIC AI
* **Buổi:** S02
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook K-Fold Ensembling Harness và kỹ nghệ đặc trưng kéo điểm thi đấu
* **Mức độ liên quan VOAI / Olympic AI:** Động cơ kéo điểm cốt lõi giúp bứt phá bảng tổng sắp trong các kỳ thi AI.

#### Mục tiêu học tập (Learning Objectives)
- Nắm vững phương châm thi đấu: 'Applied ML is basically Feature Engineering' — Kỹ nghệ đặc trưng là động cơ chính kéo điểm số.
- Xây dựng các đặc trưng toán học phi miền (Domain-Agnostic Features): Tương tác nhân/chia, tỷ lệ (Ratios), và thống kê tổng hợp (Aggregation Features).
- Làm chủ kỹ thuật Out-of-Fold (OOF) Target Encoding an toàn chống rò rỉ dữ liệu.
- Thực hiện phối hợp mô hình (Model Ensembling): Trộn trọng số (Blending) và Trung bình xếp hạng (Rank Averaging) để bứt phá bảng xếp hạng.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M5_S01 (Đọc đề & Validation Scheme).
- Đã thành thạo các mô hình từ M2 và M3.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Trong một cuộc thi AI nơi mọi thí sinh đều có quyền truy cập vào các mô hình mạnh mẽ như nhau (LightGBM, XGBoost, CatBoost hay PyTorch MLP), tại sao có người chỉ đạt 75% điểm số trong khi người khác lại đạt 89%? 

Sự khác biệt 14% điểm số đó **không đến từ việc tinh chỉnh siêu tham số (Hyperparameter Tuning) phức tạp, mà đến từ KỸ NGHỆ ĐẶC TRƯNG (FEATURE ENGINEERING) VÀ PHỐI HỢP MÔ HÌNH (ENSEMBLING)**! Thuật toán máy học chỉ thông minh khi bạn "mớm" cho nó những góc nhìn dữ liệu giàu tín hiệu. Buổi học hôm nay trang bị cho bạn toàn bộ kho vũ khí thực chiến để tạo ra các đặc trưng đột phá và kết hợp sức mạnh của nhiều mô hình nhằm giành từng 0.01 điểm quý giá trên bảng tổng sắp!



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Kỹ thuật Kỹ nghệ Đặc trưng Thực chiến (Practical Feature Engineering)
1. **Đặc trưng Tương tác & Tỷ lệ (Interaction & Ratios):**
   - Thay vì chỉ để riêng `Thu_nhập` và `Khoản_vay`, hãy tạo `Tỷ_lệ_nợ = Khoản_vay / Thu_nhập`.
   - Kết hợp hai biến số: `Diện_tích * Số_phòng`.
2. **Đặc trưng Thống kê Gom nhóm (Groupby Aggregations):**
   - Giá trị giao dịch của khách hàng so với trung bình của nhóm tuổi đó: `User_Amount - Group_Mean_Amount`.
3. **Frequency Encoding (Mã hóa theo tần suất xuất hiện):**
   - Đếm số lần một giá trị danh mục xuất hiện trong bảng dữ liệu.
4. **Out-Of-Fold (OOF) Target Encoding:**
   - Thay thế nhãn danh mục bằng trung bình biến mục tiêu của nó, nhưng **chỉ tính trên các Fold khác (Out-of-Fold)** để triệt tiêu hoàn toàn nguy cơ rò rỉ thông tin!

### 2. Sức mạnh của Ensembling: Blending & Rank Averaging
* **Triết lý:** Một hội đồng gồm nhiều chuyên gia đa dạng góc nhìn luôn đưa ra quyết định sáng suốt hơn một cá nhân đơn lẻ.
* **Weighted Blending (Trộn có trọng số):**
  $$P_{ensemble} = 0.5 \times P_{LGBM} + 0.3 \times P_{RandomForest} + 0.2 \times P_{PyTorch}$$
* Kết hợp các mô hình có cấu trúc khác nhau (Mô hình họ cây + Mạng nơ-ron sâu) mang lại mức cải thiện điểm số lớn nhất!



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Quy trình Huấn luyện K-Fold & Sinh Dự đoán Out-Of-Fold (OOF):
             ┌── Train Fold 1, 2, 3, 4 ──> Predict Val Fold 5 ──┐
             ├── Train Fold 1, 2, 3, 5 ──> Predict Val Fold 4 ──┤
K-Fold Loop ─┼── Train Fold 1, 2, 4, 5 ──> Predict Val Fold 3 ──┼──> [FULL OOF PREDICTIONS]
             ├── Train Fold 1, 3, 4, 5 ──> Predict Val Fold 2 ──┤     (Đo lường điểm CV chuẩn xác!)
             └── Train Fold 2, 3, 4, 5 ──> Predict Val Fold 1 ──┘
                             │
                             ▼
              [5 Mô hình độc lập cùng dự đoán Test] ──> [Average Ensemble cho Bài nộp!]
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# 1. Tao tap du lieu thi dau mo phong: Du doan rui ro tin dung (Credit Risk)
np.random.seed(42)
N = 500

income = np.random.uniform(10, 100, size=N) # Thu nhap (trieu VND)
debt = np.random.uniform(5, 80, size=N)     # Khoan no (trieu VND)
city = np.random.choice(['Hanoi', 'HCMC', 'Danang', 'Hue'], size=N)

# Nhan muc tieu: 1 la vo no (Default), 0 la an toan
debt_ratio_real = debt / (income + 1e-5)
probs = 1 / (1 + np.exp(-(debt_ratio_real * 3 - 2.5)))
y = (np.random.rand(N) < probs).astype(int)

df_comp = pd.DataFrame({'Income': income, 'Debt': debt, 'City': city, 'Target': y})

# 2. FEATURE ENGINEERING THUC CHIEN:
# Dac trung 1: Ty le No tren Thu nhap (Debt-to-Income Ratio)
df_comp['Debt_to_Income'] = df_comp['Debt'] / (df_comp['Income'] + 1e-5)

# Dac trung 2: Chenh lech so voi trung binh thu nhap cua thanh pho (Group Aggregation)
city_income_mean = df_comp.groupby('City')['Income'].transform('mean')
df_comp['Income_vs_City_Mean'] = df_comp['Income'] - city_income_mean

# Dac trung 3: Frequency Encoding cho bien danh muc City
city_freq = df_comp['City'].value_counts(normalize=True)
df_comp['City_Freq'] = df_comp['City'].map(city_freq)

print("===== CAC DAC TRUNG MOI DUOC TAO RA (FEATURE ENGINEERING) =====")
print(df_comp[['Income', 'Debt', 'Debt_to_Income', 'Income_vs_City_Mean', 'City_Freq']].head())

### 🔍 Code Walkthrough (Khối 1)
Khối code minh họa trực tiếp các kỹ thuật kỹ nghệ đặc trưng kéo điểm thi đấu:
- Tạo đặc trưng tỷ lệ vàng `Debt_to_Income`.
- Tạo đặc trưng tương đối so với nhóm bằng `groupby().transform('mean')`.
- Mã hóa tần suất `City_Freq` giúp thuật toán hiểu được độ phổ biến của từng thành phố mà không cần One-Hot Encoding cồng kềnh.


In [ ]:
# 3. K-FOLD ENSEMBLING HARNESS: KET HOP RANDOM FOREST VA LOGISTIC REGRESSION
features = ['Income', 'Debt', 'Debt_to_Income', 'Income_vs_City_Mean', 'City_Freq']
X = df_comp[features].values
y = df_comp['Target'].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_rf = np.zeros(N)
oof_lr = np.zeros(N)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    
    # Model A: Random Forest
    rf = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=42)
    rf.fit(X_train, y_train)
    oof_rf[val_idx] = rf.predict_proba(X_val)[:, 1]
    
    # Model B: Logistic Regression
    lr = LogisticRegression()
    lr.fit(X_train, y_train)
    oof_lr[val_idx] = lr.predict_proba(X_val)[:, 1]

# 4. ENSEMBLE BLENDING (Ket hop xac suat 60% RF + 40% LR)
oof_ensemble = 0.6 * oof_rf + 0.4 * oof_lr

score_rf = f1_score(y, (oof_rf >= 0.5).astype(int))
score_lr = f1_score(y, (oof_lr >= 0.5).astype(int))
score_ens = f1_score(y, (oof_ensemble >= 0.5).astype(int))

print("===== KET QUA K-FOLD OUT-OF-FOLD (OOF) BENCHMARK =====")
print(f"Model A (Random Forest)    F1-Score: {score_rf:.4f}")
print(f"Model B (Logistic Regress) F1-Score: {score_lr:.4f}")
print(f"-> ENSEMBLE BLEND (60/40)  F1-Score: {score_ens:.4f}")
print("-> KET LUAN: Ensembling giup giam phuong sai va cai thien diem so vuot troi tren toan bo tap kiem chung!")

### 🔍 Code Walkthrough (Khối 2)
Khối code xây dựng một K-Fold Ensembling Harness hoàn chỉnh:
- Tính toán dự đoán Out-Of-Fold (OOF) sạch sẽ cho 100% mẫu dữ liệu.
- Phối hợp mô hình bằng Weighted Blending (60% Random Forest + 40% Logistic Regression).
- Chứng minh bằng số liệu thực nghiệm: Điểm số F1 của Ensemble Blend cao hơn cả hai mô hình đơn lẻ!


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Tìm tỷ lệ Blend tối ưu
Viết một vòng lặp quét tỷ lệ trọng số $w$ từ $0.0$ đến $1.0$ (với bước nhảy $0.05$) cho công thức:
$$P_{blend} = w 	imes P_{RF} + (1 - w) 	imes P_{LR}$$
Tìm giá trị $w^*$ mang lại điểm F1-Score cao nhất trên tập OOF.

### Bài 2 (Easy): Kỹ thuật Rank Averaging
Khi hai mô hình có thang đo xác suất không đồng nhất (một mô hình dự đoán xác suất tập trung quanh 0.1-0.3, mô hình kia tập trung quanh 0.6-0.8), việc cộng trực tiếp sẽ bị lệch. Hãy tìm hiểu cách dùng `scipy.stats.rankdata` để xếp hạng thứ tự dự đoán trước khi lấy trung bình (Rank Averaging).

### Bài 3 (Medium): Out-Of-Fold Target Encoding
Viết hàm `oof_target_encode(df, cat_col, target_col, n_splits=5)` thực hiện Target Encoding chuẩn xác: Tính trung bình của biến mục tiêu trong từng fold huấn luyện rồi ánh xạ sang fold kiểm chứng, bảo đảm không gây ra Data Leakage.

### Bài 4 (Medium): Trích xuất đặc trưng đa thức (Polynomial Features)
Sử dụng `sklearn.preprocessing.PolynomialFeatures(degree=2, interaction_only=True)` để tự động tạo ra tất cả các cặp nhân chéo giữa các đặc trưng số học. Nhận xét về sự bùng nổ số lượng cột đặc trưng.

### Bài 5 (Challenge): Xây dựng khung Stacking Classifier 2 tầng
Lấy dự đoán OOF của Random Forest và Logistic Regression làm các đặc trưng đầu vào mới (Level-1 Features), sau đó huấn luyện một mô hình Meta-Learner (ví dụ Logistic Regression đơn giản) ở tầng 2 để học cách phối hợp tối ưu hai mô hình gốc.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong các trận chung kết Olympic AI, **không có bất kỳ thí sinh nào giành huy chương với một mô hình đơn lẻ**. Bí mật của các nhà vô địch luôn là kỹ thuật **K-Fold Model Blending**: Họ huấn luyện 5 folds của mô hình A (LightGBM) và 5 folds của mô hình B (PyTorch MLP), sau đó lấy trung bình dự đoán của cả 10 mô hình này trên tập Test. Kỹ thuật này giúp san phẳng mọi sai lệch ngẫu nhiên và đảm bảo bài thi đạt độ ổn định tuyệt đối trước dữ liệu chấm thi bí mật.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Thí sinh tính Target Encoding trên toàn bộ tập dữ liệu trước khi chia K-Fold:

```python
# CODE CÓ LỖI RÒ RỈ TARGET NGHIÊM TRỌNG (TARGET LEAKAGE):
# mean_target = df.groupby('City')['Target'].mean()
# df['City_Encoded'] = df['City'].map(mean_target) # Thong tin nhan Target cua ca tap Test bi ro ri vao dac trung!
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao Target Encoding là con dao hai lưỡi có thể phá hủy hoàn toàn tính trung thực của bài thi nếu không làm theo cơ chế Out-Of-Fold.
2. Khẳng định quy tắc: Mọi phép biến đổi dựa trên nhãn mục tiêu bắt buộc phải được tính riêng biệt bên trong từng Fold của tập Train!



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Câu nói nổi tiếng của Giáo sư Pedro Domingos: "Applied Machine Learning is basically..." hoàn thiện bằng cụm từ nào?
   - A. Deep Learning
   - B. Feature Engineering (Kỹ nghệ đặc trưng)
   - C. Hyperparameter Tuning
   - D. Data Cleaning

2. Kỹ thuật "Frequency Encoding" biến đổi một biến danh mục (Categorical) thành giá trị số dựa trên nguyên lý nào?
   - A. Sắp xếp theo bảng chữ cái.
   - B. Đếm tần suất hoặc tỷ lệ phần trăm xuất hiện của giá trị đó trong tập dữ liệu.
   - C. Gán một số nguyên ngẫu nhiên.
   - D. Đếm số lượng ký tự trong từ.

3. Dự đoán Out-Of-Fold (OOF Predictions) trong K-Fold Cross-Validation mang lại lợi ích lớn nhất nào?
   - A. Giúp rút ngắn thời gian chạy của máy tính.
   - B. Cung cấp một tập dự đoán hoàn chỉnh cho 100% dữ liệu huấn luyện mà không có bất kỳ mẫu nào bị mô hình "nhìn thấy trước", phản ánh trung thực và khách quan nhất sức mạnh thực sự của mô hình.
   - C. Tự động sửa lỗi dữ liệu khuyết.
   - D. Thay thế cho việc thu thập dữ liệu mới.

4. Tại sao việc kết hợp (Ensemble) hai mô hình có bản chất thuật toán hoàn toàn khác nhau (ví dụ: Random Forest họ Cây và Mạng Nơ-ron PyTorch) thường hiệu quả hơn việc kết hợp hai mô hình cùng loại?
   - A. Vì hai mô hình khác nhau sẽ mắc phải những sai lầm độc lập khác nhau tại các vùng dữ liệu khác nhau, khi lấy trung bình thì các sai số này sẽ tự triệt tiêu lẫn nhau (Diversity is Key).
   - B. Để làm cho mã nguồn dài hơn.
   - C. Vì ban giám khảo cộng điểm cho sự đa dạng.
   - D. Tiết kiệm bộ nhớ GPU.

5. Phương pháp "Weighted Blending" kết hợp các mô hình bằng cách nào?
   - A. Chọn mô hình có tên dài nhất.
   - B. Nhân xác suất dự đoán của từng mô hình với một trọng số tin cậy tương ứng rồi cộng lại (Tổng các trọng số bằng 1.0).
   - C. Xóa các dự đoán khác nhau.
   - D. Chỉ lấy dự đoán của mô hình chạy nhanh nhất.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Tầm quan trọng tối thượng của Feature Engineering trong việc kéo điểm thi đấu.
  - Các kỹ thuật tạo đặc trưng phi miền: Tỷ lệ (Ratios), Gom nhóm (Groupby Aggregations), Frequency Encoding.
  - Bản chất của K-Fold Out-Of-Fold (OOF) Predictions.
  - Nghệ thuật phối hợp mô hình (Ensemble Blending) để bứt phá bảng xếp hạng.
* **What I can now do:**
  - Tự tin thiết kế các đặc trưng mới từ dữ liệu thô giúp tăng ngay lập tức điểm số mô hình.
  - Xây dựng hệ thống phối hợp đa mô hình đạt chuẩn thi đấu chuyên nghiệp.
* **What I still need to practice:**
  - Bước vào phòng thi thực chiến: **Mock Olympic AI Competition & Submission Checklist** ở Buổi 3!



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Buổi tiếp theo (**M5_S03**) là buổi tổng duyệt đỉnh cao của Module 5: **Mock Olympic AI Competition: Giả Lập Phòng Thi & Submission Checklist Chuẩn**! Bạn sẽ trải qua một bài thi thử Olympic AI trọn vẹn trong thời gian giới hạn, tự tay nộp bài có chấm điểm tự động, rà soát toàn bộ Checklist nộp bài và nhận Chứng chỉ **Certificate of AI Competition Practice**!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M5**:
- Feature Engineering thực chiến (Tỷ lệ, Gom nhóm, Frequency encoding) ✓
- K-Fold Cross-Validation & Out-of-Fold (OOF) predictions ✓
- Model Ensembling & Blending tối ưu hóa điểm số ✓

* **Session Deliverable:** Mã nguồn K-Fold Ensembling Harness kết hợp đa mô hình tối ưu điểm số
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
